# Tct nickel occlusion

Research notebook preserved from the original experiment. See the repository README and reproduction notes before execution. Generated models and histories are written to `outputs/tct_nickel_occlusion/`.


In [ ]:
from pathlib import Path
import sys

REPOSITORY_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "research_paths.py").is_file()
)
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))
from research_paths import data_file, external_file, checkpoint_file, history_file, output_file


In [ ]:
seed_value = 777

import numpy as np
import random
import tensorflow as tf

# Set random seeds for reproducibility
np.random.seed(seed_value)
random.seed(seed_value)
tf.random.set_seed(seed_value)

# For PyTorch, if used (commented if not applicable)
try:
    import torch

    torch.manual_seed(seed_value)
    torch.cuda.manual_seed(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
except ImportError:
    pass

# Fixing thread settings to enforce determinism (optional)
import os

os.environ["TF_DETERMINISTIC_OPS"] = "1"

# Explicitly set initializers for TensorFlow/Keras models
from tensorflow.keras import initializers

initializer = initializers.GlorotUniform(seed=seed_value)


import numpy as np
import random
import tensorflow as tf

# Set random seeds for reproducibility
np.random.seed(seed_value)
random.seed(seed_value)
tf.random.set_seed(seed_value)

# For PyTorch, if used (commented if not applicable)
try:
    import torch

    torch.manual_seed(seed_value)
    torch.cuda.manual_seed(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
except ImportError:
    pass

# For scikit-learn, ensure random_state is set where applicable


import numpy as np
import random
import tensorflow as tf

# Set random seeds for reproducibility
np.random.seed(seed_value)
random.seed(seed_value)
tf.random.set_seed(seed_value)


import numpy as np
import random

# Set random seeds for reproducibility
np.random.seed(seed_value)
random.seed(seed_value)

import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import pandas as pd
import os
import csv


In [ ]:
"""
LOAD TRAINING DATA
"""

# Load training data
path = data_file("spectra/training_data_long_ontime.csv")

# Read the data from CSV
data = pd.read_csv(path, dtype=float)
col = list(data.columns)

# Extract wavelengths
wavelength = np.array([float(j) for j in col[18:1880]])

# Convert data to numpy array
data = np.array(data)

# Extract specific columns for labels
ontime = data[:, col.index("Ontime")]
conductivity = data[:, col.index("Conductivity")]
concentration = data[:, col.index("Ni")]

# Extract spectrum containing only spectrum information
spectrum = data[:, 18:1880]
spectrum = np.clip(spectrum, None, 60000)

# Indices to be removed
Pb0_index = np.array([col.index("278.249"), col.index("286.893")]) - 18
Pb1_index = np.array([col.index("360.102"), col.index("375.499")]) - 18
Pb2_index = np.array([col.index("400.609"), col.index("410.415")]) - 18
Cu_index = np.array([col.index("323.017"), col.index("340.015")]) - 18
Zn1_index = np.array([col.index("210.444"), col.index("220.089")]) - 18
Zn2_index = np.array([col.index("468.907"), col.index("485.042")]) - 18
# Collect indices to be removed
zero_index = np.concatenate(
    [
        np.arange(Pb0_index[0], Pb0_index[-1] + 1),
        np.arange(Pb1_index[0], Pb1_index[-1] + 1),
        np.arange(Pb2_index[0], Pb2_index[-1] + 1),
        np.arange(Cu_index[0], Cu_index[-1] + 1),
        np.arange(Zn1_index[0], Zn1_index[-1] + 1),
        np.arange(Zn2_index[0], Zn2_index[-1] + 1),
    ]
)

# Set specific indices to zero
spectrum[:, zero_index] = 0
spectrum = spectrum / 60000

# Dimensions and number of spectra
dimension = spectrum.shape[1]
training_number = spectrum.shape[0]

print("\n" + "=" * 40 + "\n")
print("Spectra Dimension - final", dimension)
print("Training Spectra number - final", training_number)
print("Training Data shape", spectrum.shape)
print("Training Label shape", concentration.shape)
print("\n" + "=" * 40 + "\n")


In [ ]:
"""
LOAD TESTING DATA
"""

path = data_file("spectra/testing_data_long_ontime.csv")
# data not in np.array() form (no wavelength)
data_testing = pd.read_csv(path, dtype=float)
# data in np.array() form (no wavelength)
data_testing = np.array(data_testing)
# Ontime
ontime_test = data_testing[:, col.index("Ontime")]
# Conductivity
conductivity_test = data_testing[:, col.index("Conductivity")]
# Concentration labels in ppm (target element selected below).
concentration_test = data_testing[:, col.index("Ni")]
# spectrum_test containing only spectrum information
spectrum_test = data_testing[:, 18:1880]
spectrum = np.clip(spectrum, None, 60000)
spectrum_test[:, zero_index] = 0
spectrum_test = spectrum_test / 60000
dimension = len(spectrum_test[0])
testing_number = len(spectrum_test)
print("\n" + "=" * 40 + "\n")
print("Sprctra Dimension - final", dimension)
print("Testing Spectra number - final", testing_number)
print("Testing Data shape", spectrum_test.shape)
print("Testing Label shape", concentration_test.shape)
print("\n" + "=" * 40 + "\n")


In [ ]:
plt.plot(wavelength, spectrum[0])
nonzero_count = sum(1 for w in spectrum[0] if w != 0)
print(nonzero_count)


In [ ]:
ontime = ontime.reshape(len(ontime), 1)
conductivity = conductivity.reshape(len(conductivity), 1)
concentration = concentration.reshape(len(concentration), 1)
ontime_test = ontime_test.reshape(len(ontime_test), 1)
conductivity_test = conductivity_test.reshape(len(conductivity_test), 1)
concentration_test = concentration_test.reshape(len(concentration_test), 1)
# Stack the arrays horizontally
training = spectrum
testing = spectrum_test


In [ ]:
import tensorflow.keras as tfkeras
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow.keras.backend as K
from sklearn.model_selection import train_test_split
from matplotlib.pyplot import colorbar
from sklearn.utils import shuffle
from tensorflow.keras import regularizers
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix
import numpy as np

# Training parameters
learning_rate = 0.00002
# 0.000001
opt = tfkeras.optimizers.Adam(learning_rate=learning_rate)
early_stop = tfkeras.callbacks.EarlyStopping(
    monitor="val_mae", patience=3000, verbose=0, mode="min", restore_best_weights="True"
)


In [ ]:
## from matplotlib.pyplot import colorbar1
from sklearn.utils import shuffle
from tensorflow.keras import regularizers
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from keras.layers import LSTM, Dense

dim = int(np.max(ontime) + 1)
input_spectrum = tf.keras.Input(shape=(training.shape[1],))
input_conductivity = tf.keras.Input(shape=(1,))
input_ontime = tf.keras.Input(shape=(1,))

# Reshape input_spectrum
spectrum_shape = layers.Reshape((training.shape[1], 1))(input_spectrum)
conv1 = layers.Conv1D(32, 5, kernel_regularizer=regularizers.l2(0.001))(spectrum_shape)
conv1 = layers.MaxPooling1D(3)(conv1)
conv2 = layers.Conv1D(32, 5, kernel_regularizer=regularizers.l2(0.001))(conv1)
conv2 = layers.MaxPooling1D(3)(conv2)

# Transformer Encoder Block
# Multi-head Attention
attention_output = layers.MultiHeadAttention(num_heads=4, key_dim=64)(conv2, conv2)
attention_output = layers.Dropout(0.2)(attention_output)
attention_output = layers.LayerNormalization(epsilon=1e-6)(attention_output + conv2)  # Add & Norm

# Feed-Forward Network (FFN)
ffn = layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(0.001))(
    attention_output
)
ffn = layers.Dropout(0.1)(ffn)
ffn_output = layers.LayerNormalization(epsilon=1e-6)(ffn + attention_output)  # Add & Norm

# Flatten and Dense Layers for Final Prediction
flatten = layers.Flatten()(ffn_output)
dense = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(0.001))(flatten)
dense = layers.Dropout(0.2)(dense)
dense = layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(0.001))(dense)
dense = layers.Dropout(0.2)(dense)
output = layers.Dense(1)(dense)

# Build Model
TCT_Model = Model(inputs=input_spectrum, outputs=output, name="Temporal_Convolutional_Transformer")
TCT_Model.summary()


def smape(y_true, y_pred):
    epsilon = tf.keras.backend.epsilon()  # Small constant to avoid division by zero
    numerator = tf.abs(y_pred - y_true)
    denominator = tf.abs(y_true) + tf.abs(y_pred)
    smape_loss = tf.where(
        tf.equal(y_true, 0),
        numerator,  # Use absolute error when y_true is zero
        numerator / (denominator + epsilon) * 2,  # Use SMAPE otherwise
    )
    return tf.reduce_mean(smape_loss) * 100


# Compile and train the model
TCT_Model.compile(loss=smape, optimizer=opt, metrics=["mae"])


In [ ]:
# for layer in CNN_Model.layers:
#    if layer.name.startswith("embed"):
#        transition_layer_name = layer.name
#        break
#    else:
#        continue

# transition_layer = CNN_Model.get_layer(transition_layer_name)

# Embed_Checking = tfkeras.Model(CNN_Model.input[1], transition_layer.output)
# Embed_Checking.summary()

# Embed_Checking.predict()


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import Callback, EarlyStopping


# Custom validation callback
class MultiValidationCallback(Callback):
    def __init__(self, model, validation_data_2):
        self.model = model
        self.validation_data_2 = validation_data_2

    def on_epoch_end(self, epoch, logs=None):
        X_val2, Y_val2 = self.validation_data_2
        val_preds_2 = self.model.predict(X_val2, verbose=0)

        # Calculate SMAPE and MAPE
        val_loss_2, val_mae_2 = self.model.evaluate(X_val2, Y_val2, verbose=0)
        mape_2 = np.mean(np.abs((Y_val2 - val_preds_2) / (Y_val2 + 1e-8))) * 100
        print(
            f"Epoch {epoch+1}: Validation 2 Loss = {val_loss_2:.4f}, Validation 2 MAE = {val_mae_2:.4f}, Validation 2 MAPE = {mape_2:.4f}%"
        )


# Split training and validation data
X_train, X_val, Y_train, Y_val = train_test_split(
    training, concentration, test_size=0.1, random_state=seed_value
)

# Configure the two validation datasets
validation_data_1 = (X_val, Y_val)
validation_data_2 = (testing, concentration_test)

# Early stopping based on validation_data_1
early_stop = EarlyStopping(
    monitor="val_loss",  # Monitor loss on the first validation dataset
    patience=500,  # Stop after the configured patience without improvement
    restore_best_weights=True,  # Restore the best model weights
)

# Initialize the callback
multi_val_callback = MultiValidationCallback(model=TCT_Model, validation_data_2=validation_data_2)

# Train the model
train_history = TCT_Model.fit(
    X_train,
    Y_train,
    validation_data=validation_data_1,  # Use the first dataset for internal validation
    batch_size=32,
    epochs=20000,
    verbose=2,
    callbacks=[early_stop, multi_val_callback],  # Include early stopping and custom validation
)

# Record training and validation history
Epoch = len(train_history.history["mae"])
Training_History = np.zeros((4, Epoch))
Training_History[0, :] = np.array(train_history.history["loss"])
Training_History[1, :] = np.array(train_history.history["mae"])
Training_History[2, :] = np.array(train_history.history["val_loss"])
Training_History[3, :] = np.array(train_history.history["val_mae"])

# Save training history
Array_name = str(output_file("Cu_Transfer.npy", "tct_nickel_occlusion"))
np.save(Array_name, Training_History)

# Save the model
Model_name = str(output_file("Cu_Transfer.h5", "tct_nickel_occlusion"))
TCT_Model.save(Model_name)


In [ ]:
# Training
Y_pred = TCT_Model.predict([training])
plt.plot(Y_pred, ".")
plt.plot(concentration)
plt.xlabel("number of training data")
plt.ylabel("ppm")
plt.show()
print(tf.keras.losses.MeanSquaredError()(concentration, Y_pred))


In [ ]:
plt.plot(Training_History[1, :], color="blue", label="Training mae")
plt.plot(Training_History[3, :], color="red", label="Val mae")
plt.legend()
plt.xlabel("epochs")
plt.ylabel("mae")
plt.show()
Y_pred = TCT_Model.predict([testing])
print(tf.keras.losses.MeanSquaredError()(concentration_test, Y_pred))

# Testing
plt.plot(range(0, 30), Y_pred[:30], ".", color="orange", alpha=0.7)
plt.plot(range(30, 60), Y_pred[30:60], ".", color="m", alpha=0.7)
plt.plot(range(60, 90), Y_pred[60:90], ".", color="orange", alpha=0.7)
plt.plot(range(90, 120), Y_pred[90:120], ".", color="m", alpha=0.7)
plt.plot(range(120, 150), Y_pred[120:150], ".", color="orange", alpha=0.7)
plt.plot(range(150, 180), Y_pred[150:180], ".", color="m", alpha=0.7)
plt.plot(range(180, 210), Y_pred[180:210], ".", color="orange", alpha=0.7)
plt.plot(range(210, 240), Y_pred[210:240], ".", color="m", alpha=0.7)
plt.plot(range(240, 270), Y_pred[240:270], ".", color="orange", alpha=0.7)
plt.plot(range(270, 300), Y_pred[270:300], ".", color="m", alpha=0.7)

plt.plot(concentration_test, color="black", linewidth=3.0)
plt.xlabel("number of testing data", fontsize=15)
plt.ylabel("Concentration (ppm)", fontsize=15)
plt.title("Predictions vs Actual Concentrations", fontsize=20)
plt.show()
Y_pred = TCT_Model.predict([testing])
print(tf.keras.losses.MeanSquaredError()(concentration_test, Y_pred))


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import colorbar
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
import tensorflow.keras.backend as K
import scipy.io as sio
from keras import activations
from matplotlib import cm
from sklearn.metrics import confusion_matrix

import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

cmap = "viridis"


def color_map(data, cmap):
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    cmo = plt.cm.get_cmap(cmap)
    cs, k = list(), 256 / cmo.N
    k = int(k)

    for i in range(cmo.N):
        c = cmo(i)
        for j in range(int(i * k), int(i + 1) * k):
            cs.append(c)
    cs = np.array(cs)
    data = np.uint8(255 * (data - dmin) / (dmax - dmin))

    return cs[data]


n = 235
Pk = TCT_Model.predict(np.array([spectrum[n]]))
print(Pk)
WS = []
for i in range(20):
    WS.append(5 * (i + 1))
# Store the prediction from each occlusion experiment
Pijks = np.zeros((20, 400))  # Rows: window sizes; columns: window positions
Pijks_PK = np.zeros((20, 400))  # Rows: window sizes; columns: window positions

WL = wavelength
for m in range(len(WS)):  # Iterate over occlusion window sizes
    for t in range(
        int(len(WL) / WS[m])
    ):  # Occlude successive wavelength regions with this window size
        C_data = np.copy(
            np.array([spectrum[n]])
        )  # Copy the spectrum to preserve the original input
        C_data[:, WS[m] * t : WS[m] * (t + 1)] = (
            0  # Zero the selected wavelength region in the copied spectrum
        )
        Pijk = TCT_Model.predict(np.array(C_data))  # Predict from the occluded spectrum
        Pijks[m, t] = Pijk  # Record the prediction
        Pijks_PK[m, t] = Pijk - Pk  # Record the change from the unoccluded prediction
        # print(m,t)

# Aggregate weighted importance across spectral windows
ORSFE_total = np.zeros(1862)  # Accumulate importance across window sizes
ORSFE_eachW = np.zeros(1862)  # Store per-wavelength importance for one window size

m = len(WS)  # Number of window sizes
for m in range(m):  # Select a window size
    weights = np.log2(np.max(WS)) / np.log2(
        WS[m]
    )  # Weight this window size, assigning larger weights to smaller windows
    t = int(len(WL) / WS[m])  # Number of complete window positions across the spectrum
    for t in range(t):
        ORSFE_eachW[WS[m] * t : WS[m] * (t + 1)] = (
            -Pijks_PK[m, t] * weights * np.ones(WS[m])
        )  # Assign weighted importance to the corresponding wavelength region
    ORSFE_total += ORSFE_eachW  # Accumulate importance from different window sizes
ORSFE_total /= np.max(ORSFE_total)  # Scale the maximum importance to one
WL = np.array(wavelength, dtype=float)
x = WL  # Wavelengths on the horizontal axis
y = spectrum[n]  # Input spectral intensity on the vertical axis
ps = np.stack((x, y), axis=1)
segments = np.stack((ps[:-1], ps[1:]), axis=1)
###########################################################################
# Each adjacent pair of wavelength points forms one line segment.    #
# Segments have shape (number of segments, 2 endpoints, 2 coordinates).                     #
# For example, consecutive segments share an endpoint.    #
#                         x2 , y2]                           x3 , y3]   #
#########################################################################

# Map importance weights to colors
colors = color_map(ORSFE_total, cmap)
# Create a line collection with the specified colors, widths, and styles.
line_segments = LineCollection(segments, colors=colors, linewidths=1.5, linestyles="-", cmap=cmap)

# Display the figure
fig, ax = plt.subplots()
ax.set_xlim((float(min(x)) - 10, float(max(x)) + 10))
ax.set_ylim((0, max(y) + 0.05))
ax.add_collection(line_segments)
sm = plt.cm.ScalarMappable(cmap="viridis")
sm.set_array([])
cb = fig.colorbar(sm, cmap="viridis")
plt.show()


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import matplotlib.colors as mcolors
import tensorflow as tf

# -----------------------------------------------------------
# 1. Build an M x N heatmap with one row per window size
# -----------------------------------------------------------
heatmap_data = np.zeros((len(WS), len(WL)))
for i, ws in enumerate(WS):
    for t in range(int(len(WL) / ws)):
        start = ws * t
        end = ws * (t + 1)
        heatmap_data[i, start:end] = ORSFE_total[start:end]

# -----------------------------------------------------------
# 2. Set the color limits and colormap
# -----------------------------------------------------------
all_data = np.concatenate([heatmap_data.ravel(), ORSFE_total])
vmin, vmax = np.nanmin(all_data), np.nanmax(all_data)

common_norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
cmap_name = "viridis"
cmap_obj = plt.cm.get_cmap(cmap_name)

# -----------------------------------------------------------
# 3. Create two subplots and reserve space on the right for a colorbar
# -----------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(10, 6), sharex=False, gridspec_kw={"height_ratios": [1, 3]}
)

# Use 85% of the figure width for the subplots.
# Reserve the remaining 15% for the colorbar.
fig.subplots_adjust(right=0.85)

# -----------------------------------------------------------
# 4. Draw the heatmap on ax1
#    Hide the axes on the upper subplot
# -----------------------------------------------------------
Y_idx = np.arange(len(WS))  # shape=(M,)
X, Y = np.meshgrid(WL, Y_idx)

hmap = ax1.pcolormesh(X, Y, heatmap_data, cmap=cmap_obj, norm=common_norm, shading="auto")

# Hide spines, ticks, tick labels, and axis labels
ax1.set_xlabel("")
ax1.set_ylabel("")
ax1.tick_params(axis="both", labelsize=14)
ax1.set_yticks([])
for spine in ["top", "right", "left", "bottom"]:
    ax1.spines[spine].set_visible(False)

# -----------------------------------------------------------
# 5. Draw the colored spectrum and overlay a reference trace on ax2
# -----------------------------------------------------------
x = WL
y = spectrum[n]
ps = np.stack((x, y), axis=1)
segments = np.stack((ps[:-1], ps[1:]), axis=1)

line_colors = cmap_obj(common_norm(ORSFE_total))
line_segments = LineCollection(segments, colors=line_colors, linewidths=1.5, linestyles="-")
# Uncomment the following line to add the colored segments
# ax2.add_collection(line_segments)

# Overlay a single-color reference trace
ax2.plot(x, y, color="k", linewidth=0.8)

# Axis limits
ax2.set_xlim([x.min(), x.max()])
ax2.set_ylim([0, y.max() * 1.05])

# Increase the axis tick-label font size
ax2.tick_params(axis="both", labelsize=14)

# Set axis labels and font sizes
ax2.set_xlabel("Wavelength (nm)", fontsize=20)
ax2.set_ylabel("Intensity", fontsize=20)

# -----------------------------------------------------------
# 6. Add a colorbar in a custom axis on the right
# -----------------------------------------------------------
cbar_ax = fig.add_axes([0.88, 0.1, 0.02, 0.75])
sm = plt.cm.ScalarMappable(norm=common_norm, cmap=cmap_name)
sm.set_array([])  # Display the colormap without a data array
cb = plt.colorbar(sm, cax=cbar_ax, orientation="vertical")

# Increase the colorbar tick-label and label sizes
cb.ax.tick_params(labelsize=16)
cb.set_label("Feature Importance", fontsize=20)

# -----------------------------------------------------------
# 7. Display the figure
#   When using tight_layout, set rect to reserve space for the colorbar
# -----------------------------------------------------------
plt.show()
